# Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from datasets import Dataset
from peft import LoraConfig, TaskType
import xgboost as xgb
import torch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

2026-05-08 14:00:24.599444: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-08 14:00:24.647562: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/dkusmenko/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving

In [2]:
import os
# This enables memory fragmentation handling specifically for AMD HIP
os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import torch
def print_gpu_utilization():
    if not torch.cuda.is_available():
        print("No GPU detected.")
        return

    # On AMD ROCm, 'cuda' functions query the HIP backend
    reserved = torch.cuda.memory_reserved()
    allocated = torch.cuda.memory_allocated()
    total_memory = torch.cuda.get_device_properties(0).total_memory
    
    print(f"Total GPU Mem: {total_memory / 1024**3:.2f} GB")
    print(f"Reserved (Cached): {reserved / 1024**3:.2f} GB")
    print(f"Allocated (Active): {allocated / 1024**3:.2f} GB")
    print(f"Free (Approx): {(total_memory - reserved) / 1024**3:.2f} GB")
    print("-" * 30)

# Run it
print_gpu_utilization()

Total GPU Mem: 15.82 GB
Reserved (Cached): 0.00 GB
Allocated (Active): 0.00 GB
Free (Approx): 15.82 GB
------------------------------


/home/dkusmenko/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


# Data Import, Clean, and Sample

In [4]:
# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
# Load in freMTPL2freq dataset
dataset = fetch_openml(data_id=41214, as_frame=True)
full_df = dataset.frame

# Clean basic types first
full_df['ClaimNb'] = pd.to_numeric(full_df['ClaimNb'])
full_df['Exposure'] = pd.to_numeric(full_df['Exposure'])
full_df['Exposure'] = full_df['Exposure'].clip(upper=1.0)
full_df['Frequency'] = full_df['ClaimNb'] / full_df['Exposure']

# mapping for contextualized factors
brand_mapping = {'B1': 'Renault, Nissan, or Citroen', 'B2': 'Renault, Nissan, or Citroen',
                 'B3': 'Volkswagen, Audi, Skoda, or Seat', 'B4': 'Opel, General Motors, or Ford',
                 'B5': 'Opel, General Motors, or Ford','B6': 'Fiat', 'B10':'Mercedes, Chrysler, or BMW',
                 'B11':'Mercedes, Chrysler, or BMW', 'B12': 'Japanese (except Nissan) or Korean', 'B13': 'Other','B14': 'Other' }

region_mapping = {
    "R11": "Île-de-France",
    "R21": "Champagne-Ardenne",
    "R22": "Picardie",
    "R23": "Haute-Normandie",
    "R24": "Centre",
    "R25": "Basse-Normandie",
    "R26": "Bourgogne",
    "R31": "Nord–Pas-de-Calais",
    "R41": "Lorraine",
    "R42": "Alsace",
    "R43": "Franche–Comté",
    "R52": "Pays de la Loire",
    "R53": "Bretagne",
    "R54": "Poitou–Charentes",
    "R72": "Aquitaine",
    "R73": "Midi–Pyrénées",
    "R74": "Limousin",
    "R82": "Rhône–Alpes",
    "R83": "Auvergne",
    "R91": "Languedoc–Roussillon",
    "R93": "Provence–Alpes–Côte d’Azur",
    "R94": "Corse"
}

area_mapping = {
    "A": "rural area",
    "B": "semi-rural area",
    "C": "suburban-fringe area",
    "D": "suburban area",
    "E": "urban area",
    "F": "urban center"
}

gas_mapping = {
    "'Diesel'": "Diesel",
    "'Regular'": "Regular"

}

full_df["VehBrand"] = full_df["VehBrand"].map(brand_mapping)
full_df["Region"] = full_df["Region"].map(region_mapping)
full_df["Area"] = full_df["Area"].map(area_mapping)
full_df["VehGas"] = full_df["VehGas"].map(gas_mapping)


In [5]:
import pandas as pd
import numpy as np

# 1. Define the list of 10 occupations
occupations = [
    'Delivery Driver', 'Office Worker', 'Retired', 'Student', 'Actuary', 
    'Professor', 'Lawyer', 'Nurse', 'Mechanic', 'Construction Worker'
]

# 2. Assign randomly to a new column in your DataFrame (e.g., 'df')
# We use np.random.seed for reproducibility
np.random.seed(42)

# Assuming your main dataframe is named 'df'
full_df['Occupation'] = np.random.choice(occupations, size=len(full_df))

# 3. Verify the distribution
print(full_df['Occupation'].value_counts())

Occupation
Actuary                68073
Student                67985
Retired                67976
Office Worker          67854
Delivery Driver        67839
Lawyer                 67771
Construction Worker    67746
Professor              67724
Nurse                  67549
Mechanic               67496
Name: count, dtype: int64


In [6]:
# Load the split indices
df_splits = pd.read_csv('freMTPL2freq_split_indices.csv')

# Ensure IDpol is the same type in both dataframes for a clean merge
full_df['IDpol'] = full_df['IDpol'].astype(int)
df_splits['IDpol'] = df_splits['IDpol'].astype(int)

# Merge the dataset with the split indicators
# We use a left join to keep the original data rows
df_merged = full_df.merge(df_splits, on='IDpol', how='left')

# Create the subsets based on the indicator columns
train_df = df_merged[df_merged['is_train'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
test_df = df_merged[df_merged['is_test'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()
finetune_df = df_merged[df_merged['is_finetune'] == 1].drop(columns=['is_train', 'is_test', 'is_finetune']).copy()

# Print results
print(f"Total rows: {len(full_df)}")
print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Finetune rows: {len(finetune_df)}")

# Inspect the test set
print(test_df.head())

Total rows: 678013
Train rows: 500000
Test rows: 100000
Finetune rows: 76782
    IDpol  ClaimNb  Exposure                  Area  VehPower  VehAge  DrivAge  \
0       1        1      0.10         suburban area         5       0       55   
6      15        1      0.45            urban area         6       2       38   
22     50        1      0.06            urban area         7       0       73   
24     53        1      0.55         suburban area         5       0       33   
27     58        1      0.03  suburban-fringe area         5       0       59   

    BonusMalus                            VehBrand   VehGas  Density  \
0           50  Japanese (except Nissan) or Korean  Regular     1217   
6           50  Japanese (except Nissan) or Korean  Regular     3003   
22          50  Japanese (except Nissan) or Korean  Regular     3317   
24         100  Japanese (except Nissan) or Korean  Regular     1746   
27          50  Japanese (except Nissan) or Korean  Regular      455   

   

# Create Prompts

In [7]:
# ==========================================
# 2. SERIALIZATION (Tabular -> Text)
# ==========================================

def serialize_row_occupation(row):
    """
    Converts a row of insurance covariates into a natural language prompt.
    Uses a fixed template for consistency between Training and Inference.
    """
    # Handling categorical values cleanly
    veh_brand = str(row['VehBrand']).strip()
    veh_gas = str(row['VehGas']).strip()
    area = str(row['Area']).strip()
    region = str(row['Region']).strip()
    
    return (f"""You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: {row['DrivAge']} years old (in France people can drive starting at age 18)
- Land Type: {area}
- Region: {region}, France
- Population density: {row['Density']} people/km2 (average density is 1792 people/km2)
- Vehicle: {veh_brand}
- Vehicle Age: {row['VehAge']} years old
- Fuel type: {veh_gas} (either Diesel or Regular Gasoline)
- Power class: {row['VehPower']} (min = 4, max = 15)
- Bonus-Malus score: {row['BonusMalus']} (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)
- Occupation: {row['Occupation']}"""
    )


train_df['text_desc_occu'] = train_df.apply(serialize_row_occupation, axis=1)

test_df['text_desc_occu'] = test_df.apply(serialize_row_occupation, axis=1)


### Example Prompt

You are an auto insurance underwriter. Evaluate the risk level of a policyholder based strictly on the following insurance-related information:
- Policyholder Age: 78 years old (in France people can drive starting at age 18)
- Land Type: rural area
- Region: Poitou–Charentes, France
- Population density: 43 people/km2 (average density is 1792 people/km2)
- Vehicle: Renault, Nissan, or Citroen
- Vehicle Age: 15 years old
- Fuel type: Regular (either Diesel or Regular Gasoline)
- Power class: 7 (min = 4, max = 15)
- Bonus-Malus score: 50 (scored between 50 and 230 with entrance level 100, <100 means bonus, >100 means malus)
- Occupation: Student

# Embedding Generation

# Using Base Qwen Model

In [8]:
# Initialize Model
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B", 
    trust_remote_code=True, 
    device="cuda",
    model_kwargs={
        "torch_dtype": torch.float16,   # Critical for speed/VRAM
        "attn_implementation": "sdpa"   # Faster attention
    }
)

`torch_dtype` is deprecated! Use `dtype` instead!
/home/dkusmenko/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at /pytorch/c10/hip/HIPAllocatorConfig.h:29.)
  return t.to(


In [9]:
# Test Model works
print("Model loaded successfully!")
with model.truncate_sentence_embeddings(truncate_dim=64):
    embeddings_truncated = model.encode(["hello there", "hiya"])
assert embeddings_truncated.shape[-1] == 64

Model loaded successfully!


/usr/lib/python3.10/contextlib.py:103: FutureWarning: The `truncate_sentence_embeddings` method has been renamed to `truncate_embeddings`.
  self.gen = func(*args, **kwds)
/home/dkusmenko/.local/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:256.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/dkusmenko/.local/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:302.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [10]:
embeddings_truncated

array([[-1.8784e-02, -5.9357e-03, -9.9487e-03, -4.6478e-02, -3.4499e-04,
        -2.1687e-03, -4.1290e-02,  2.6505e-02, -9.2468e-02,  1.5198e-02,
         2.1820e-03, -2.9907e-02,  1.0095e-01, -8.7357e-03, -4.7638e-02,
         7.6416e-02, -2.5848e-02,  6.2225e-02,  1.0931e-01, -8.1299e-02,
         3.3081e-02,  9.8038e-03, -3.9032e-02,  1.3342e-01,  5.8842e-04,
        -7.8583e-03, -3.0991e-02,  1.0962e-01,  1.2772e-02, -3.5858e-03,
         1.7761e-02,  3.5187e-02, -1.1078e-02,  4.0169e-03, -3.4760e-02,
        -1.2779e-02, -1.7044e-02, -1.1711e-03, -2.9800e-02,  5.6976e-02,
        -1.7181e-02,  2.9373e-02,  6.8726e-02, -1.9272e-02, -6.6638e-05,
        -3.1174e-02,  5.5084e-02, -1.4748e-02,  1.0414e-03, -9.4604e-03,
        -2.7710e-02, -2.5681e-02,  1.4664e-02,  8.5526e-03,  9.0714e-03,
        -5.8838e-02,  5.4932e-02, -3.5591e-03,  3.7079e-02, -8.3542e-03,
        -9.2957e-02,  4.6600e-02, -2.4231e-02,  9.5062e-03],
       [-1.6165e-03,  1.8890e-02, -1.5717e-02, -5.3925e-02,  2.

Model is outputting embeddings of dim 64

### Check VRAM Usage

In [11]:
print_gpu_utilization()

Total GPU Mem: 15.82 GB
Reserved (Cached): 1.21 GB
Allocated (Active): 1.11 GB
Free (Approx): 14.61 GB
------------------------------


Now want to generate embeddings from data

Ensure model is running on GPU

In [12]:
print(model.device)

cuda:0


In [13]:
import torch

# 1. Define the device (On AMD ROCm, we still call it 'cuda' in PyTorch)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Moving model to: {device}")

# 2. Move the model
model = model.to(device)

Moving model to: cuda


In [14]:
print("Generating embeddings for GLM...")

train_occu = model.encode(train_df['text_desc_occu'].tolist(), batch_size=64, show_progress_bar=True)

test_occu = model.encode(test_df['text_desc_occu'].tolist(), batch_size=64, show_progress_bar=True)


Generating embeddings for GLM...


Batches:   0%|          | 0/7813 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [15]:
import numpy as np

# Save everything in one single archive
np.savez(
    "embeddings/train_occupation.npz", 
    X=train_occu,           # The Features (Embeddings)
    y=train_df['ClaimNb'].values, # The Target (Counts)
    w=train_df['Exposure'].values # The Weight (Exposure)
)

print("Saved all testing data to embeddings/train_occupation.npz")

np.savez(
    "embeddings/test_occupation.npz", 
    X=test_occu,           # The Features (Embeddings)
    y=test_df['ClaimNb'].values, # The Target (Counts)
    w=test_df['Exposure'].values # The Weight (Exposure)
)

print("Saved all testing data to embeddings/test_occupation.npz")

Saved all testing data to embeddings/train_occupation.npz
Saved all testing data to embeddings/test_occupation.npz


# Downstream GLM

In [2]:
import numpy as np

train_df = np.load("embeddings/train_baseqwen.npz")

test_df = np.load("embeddings/test_baseqwen.npz")

train_occu = np.load("embeddings/train_occupation.npz")

test_occu = np.load("embeddings/test_occupation.npz")

In [3]:
import numpy as np
from scipy.spatial.distance import cdist

def mean_cosine_distance(base_df, diff_df1, batch_size=1000):
    """
    Compute the mean cosine distance using batching to avoid MemoryError.
    """
    # Ensure inputs are numpy arrays for speed
    base = np.array(base_df.tolist()) if hasattr(base_df, 'tolist') else np.array(base_df)
    target = np.array(diff_df1.tolist()) if hasattr(diff_df1, 'tolist') else np.array(diff_df1)
    
    total_sum = 0.0
    total_count = len(base) * len(target)
    
    # Process base_df in chunks
    for i in range(0, len(base), batch_size):
        chunk = base[i : i + batch_size]
        # Calculate pairwise distances for this chunk only
        # Resulting shape: (batch_size, len(target))
        distances = cdist(chunk, target, metric="cosine")
        total_sum += np.sum(distances)
        
    return total_sum / total_count

# Call the function
result = mean_cosine_distance(test_df['X'][:5000], test_occu['X'][:5000])
print(f"Mean Cosine Distance: {result}")

Mean Cosine Distance: 0.059820799458985


In [4]:
# Partition Data
X_train = train_df['X']
meta_train = [train_df['y'], train_df['w']]

meta_train = pd.DataFrame({
    'ClaimNb': meta_train[0],
    'Exposure': meta_train[1]
})


X_test = test_df['X']
meta_test = [test_df['y'], test_df['w']]

meta_test = pd.DataFrame({
    'ClaimNb': meta_test[0],
    'Exposure': meta_test[1]
})


# Setup Scaler & PCA with Pandas Output
scaler = StandardScaler().set_output(transform='pandas')
pca = PCA(n_components=48).set_output(transform='pandas')

# Pipeline Execution
# TRAIN: Fit & Transform
# Scaler returns a DF with index preserved -> PCA returns a DF with index preserved
X_train_scaled = scaler.fit_transform(X_train)
X_train_pca = pca.fit_transform(X_train_scaled)

# TEST: Transform Only
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)

# Reconstruction
# Because indices are preserved, pandas aligns rows automatically.
# We can also rename columns cleanly if we want "PC1" instead of "pca0"

# Rename columns from 'pca0' to 'PC1', 'PC2'...
new_col_names = [f"PC{i+1}" for i in range(48)]
X_train_pca.columns = new_col_names
X_test_pca.columns = new_col_names


# Concatenate (Join)
final_train = pd.concat([X_train_pca, meta_train], axis=1)
final_test = pd.concat([X_test_pca, meta_test], axis=1)


print(f"Original Predictors: {len(X_train)}")
print(f"Reduced Predictors:  {X_train_pca.shape[1]}")
print(f"Final Train Shape:   {final_train.shape}")
print(final_train.head())

Original Predictors: 500000
Reduced Predictors:  48
Final Train Shape:   (500000, 50)
         PC1        PC2        PC3        PC4       PC5        PC6       PC7  \
0  15.310307   8.990679 -10.267797   2.634130 -3.456980   4.424897 -0.820731   
1  13.361327  11.378126  12.057166   9.178955 -2.067735   5.622818  1.206991   
2  13.407091  11.418993  12.020130   9.199942 -2.113281   5.646997  1.195243   
3  18.563196  14.571923   6.537592  -2.279069  3.239272 -10.736015  8.748997   
4  16.671796   7.289340  -7.038250  13.610195 -1.339093  -2.388616  0.027084   

        PC8        PC9       PC10  ...      PC41      PC42      PC43  \
0 -4.974869 -15.895921  10.834917  ...  0.981171 -0.373009  0.744505   
1  1.495947   3.993313  -1.292162  ... -1.997557  0.286662  2.366450   
2  1.497379   3.972074  -1.253709  ... -2.012132  0.278498  2.366816   
3 -4.289826  -0.417671   1.058828  ...  1.117044 -0.837260 -0.733189   
4  0.733989   5.359518 -11.200463  ... -2.531343 -0.806651  0.279537   



In [5]:
import statsmodels.api as sm

# Add constant (intercept) to PCA features
X_train_glm = sm.add_constant(X_train_pca)
X_test_glm  = sm.add_constant(X_test_pca)



# Fit Poisson GLM with Exposure offset
# offset = log(Exposure) is standard for claim frequency modelling
offset_train = np.log(final_train['Exposure'])
offset_test  = np.log(final_test['Exposure'])

poisson_model = sm.GLM(
    final_train['ClaimNb'],
    X_train_glm,
    family=sm.families.Poisson(),
    offset=offset_train
).fit()

print(poisson_model.summary())

# Predict on test set (returns expected claim counts)
y_pred = poisson_model.predict(X_test_glm, offset=offset_test)



# Evaluate: Poisson deviance is the standard metric for claim frequency models
from sklearn.metrics import mean_poisson_deviance

y_test = final_test['ClaimNb']
exp_test = final_test['Exposure']


# Predicted frequency (claims per unit exposure) for deviance calculation
freq_pred = y_pred / exp_test
freq_actual = y_test / exp_test

deviance = mean_poisson_deviance(freq_actual, freq_pred)
print(f"Mean Poisson Deviance for No Occupation: {deviance:.6f}")


                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               500000
Model:                            GLM   Df Residuals:                   499951
Model Family:                 Poisson   Df Model:                           48
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0580e+05
Date:                Fri, 08 May 2026   Deviance:                   1.6081e+05
Time:                        14:01:49   Pearson chi2:                 1.19e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.006792
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.2974      0.006   -362.518      0.0

In [6]:
# Partition Data
X_train = train_occu['X']
meta_train = [train_occu['y'], train_occu['w']]

meta_train = pd.DataFrame({
    'ClaimNb': meta_train[0],
    'Exposure': meta_train[1]
})


X_test = test_occu['X']
meta_test = [test_occu['y'], test_occu['w']]

meta_test = pd.DataFrame({
    'ClaimNb': meta_test[0],
    'Exposure': meta_test[1]
})


# Setup Scaler & PCA with Pandas Output
scaler = StandardScaler().set_output(transform='pandas')
pca = PCA(n_components=48).set_output(transform='pandas')

# Pipeline Execution
# TRAIN: Fit & Transform
# Scaler returns a DF with index preserved -> PCA returns a DF with index preserved
X_train_scaled = scaler.fit_transform(X_train)
X_train_pca = pca.fit_transform(X_train_scaled)

# TEST: Transform Only
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)

# Reconstruction
# Because indices are preserved, pandas aligns rows automatically.
# We can also rename columns cleanly if we want "PC1" instead of "pca0"

# Rename columns from 'pca0' to 'PC1', 'PC2'...
new_col_names = [f"PC{i+1}" for i in range(48)]
X_train_pca.columns = new_col_names
X_test_pca.columns = new_col_names


# Concatenate (Join)
final_train = pd.concat([X_train_pca, meta_train], axis=1)
final_test = pd.concat([X_test_pca, meta_test], axis=1)


print(f"Original Predictors: {len(X_train)}")
print(f"Reduced Predictors:  {X_train_pca.shape[1]}")
print(f"Final Train Shape:   {final_train.shape}")
print(final_train.head())

Original Predictors: 500000
Reduced Predictors:  48
Final Train Shape:   (500000, 50)
         PC1       PC2        PC3        PC4       PC5       PC6       PC7  \
0 -19.430484  5.278952   8.199012  -3.983055 -1.966059  6.589604  3.752286   
1 -16.993003  6.214939 -15.008565  -4.769916 -3.024225  3.945669 -0.167378   
2 -17.514796  6.733992 -14.441476  -4.369976 -2.203252  4.354646 -0.791989   
3 -23.523225  8.560843  -3.927216   7.217937  3.279348 -3.687068 -7.629715   
4 -20.067819  2.097820   0.865850 -13.908370  0.820649 -0.836225 -3.474919   

        PC8        PC9      PC10  ...      PC41      PC42      PC43      PC44  \
0  4.044699 -12.205359 -8.201050  ...  1.989475 -1.733020  1.027199  1.894238   
1 -0.559217   2.539123  3.099586  ... -0.291413 -2.834971  4.487054 -2.138485   
2 -0.067335   2.098891  3.177492  ...  0.875642 -1.905703  4.077463 -0.720419   
3  9.211762   1.246988 -1.662022  ... -2.844541  1.795555 -0.339239 -1.635426   
4 -0.350411   2.203011  3.822088  ... -0

In [7]:
import statsmodels.api as sm

# Add constant (intercept) to PCA features
X_train_glm = sm.add_constant(X_train_pca)
X_test_glm  = sm.add_constant(X_test_pca)



# Fit Poisson GLM with Exposure offset
# offset = log(Exposure) is standard for claim frequency modelling
offset_train = np.log(final_train['Exposure'])
offset_test  = np.log(final_test['Exposure'])

poisson_model = sm.GLM(
    final_train['ClaimNb'],
    X_train_glm,
    family=sm.families.Poisson(),
    offset=offset_train
).fit()

print(poisson_model.summary())

# Predict on test set (returns expected claim counts)
y_pred = poisson_model.predict(X_test_glm, offset=offset_test)



# Evaluate: Poisson deviance is the standard metric for claim frequency models
from sklearn.metrics import mean_poisson_deviance

y_test = final_test['ClaimNb']
exp_test = final_test['Exposure']


# Predicted frequency (claims per unit exposure) for deviance calculation
freq_pred = y_pred / exp_test
freq_actual = y_test / exp_test

deviance = mean_poisson_deviance(freq_actual, freq_pred)
print(f"Mean Poisson Deviance for Occupation Augmentation: {deviance:.6f}")


                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               500000
Model:                            GLM   Df Residuals:                   499951
Model Family:                 Poisson   Df Model:                           48
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0598e+05
Date:                Fri, 08 May 2026   Deviance:                   1.6116e+05
Time:                        14:02:31   Pearson chi2:                 1.23e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.006100
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.2958      0.006   -363.338      0.0